# 04 – NLP / RAG: Car Advisor Chatbot

**Task:** Build a Retrieval-Augmented Generation (RAG) chatbot that explains car predictions and answers car-related questions in natural language.

**Input:** FAISS vector store built from our used-car knowledge base; predictions from Block 1 (CV brand classifier) and Block 2 (ML price predictor).

**Output:** Natural language explanations, buying advice, and a unified `full_pipeline()` function that ties all three blocks together.

**Pipeline summary:**
```
Image    ──►  ResNet18        ──►  Brand
Features ──►  Random Forest   ──►  Price
Query    ──►  FAISS retrieve  ──►  GPT-3.5-turbo  ──►  Explanation
```

---
## Section 1 – Setup & Imports

We start by loading all required libraries and configuring the environment.

- **python-dotenv** loads the `OPENAI_API_KEY` from `.env` so we never hard-code secrets.
- **sentence-transformers** provides a lightweight local model to convert text into dense vector embeddings.
- **faiss-cpu** stores and searches those embeddings at scale using approximate nearest-neighbour search.
- **openai** provides access to GPT-3.5-turbo for natural language generation.

In [ ]:
import os
import json
import glob
import warnings
import numpy as np
import pandas as pd

import openai
from openai import OpenAI

import faiss
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

import joblib
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# Load environment variables from .env (must contain OPENAI_API_KEY)
load_dotenv(dotenv_path='../.env')

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('OPENAI_API_KEY not found. Add it to .env in the project root.')

client = OpenAI(api_key=api_key)
print('OpenAI client initialised.')

In [ ]:
# Load the processed used-cars dataset
df = pd.read_csv('../data/processed/used_cars_clean.csv')

# Fix boolean columns stored as strings by pandas get_dummies
bool_cols = [c for c in df.columns if c.startswith(('Fuel_Type_', 'Transmission_', 'Owner_Type_'))]
df[bool_cols] = df[bool_cols].replace({'True': 1, 'False': 0}).astype(int)

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Price range: {df["Price"].min():.1f} to {df["Price"].max():.1f} Lakh INR')
df.head(3)

In [ ]:
# Quick connectivity test: send a simple message to GPT-3.5-turbo
try:
    test_response = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': 'Hello! Reply with just: Connection successful.'}],
        max_tokens=20,
        temperature=0
    )
    test_text = test_response.choices[0].message.content.strip()
    print(f'OpenAI API test response: {test_text}')
    print('OpenAI connection confirmed.')
except Exception as e:
    print(f'OpenAI API error: {e}')

---
## Section 2 – Build Knowledge Base for RAG

### What is a knowledge base?

A **knowledge base** is a curated collection of text documents that contain factual information relevant to our domain. In a RAG system, the LLM does not answer questions from its training memory alone — instead it first retrieves the most relevant documents from our knowledge base, then generates an answer grounded in that retrieved context.

**Why build our own?** Because:
1. GPT-3.5-turbo knows about cars in general but has no knowledge of *our specific dataset* (prices, brands, statistics).
2. We can inject domain facts (e.g. average BMW price in our data) without fine-tuning the model.
3. Retrieval keeps answers accurate and up-to-date — when the dataset changes, just rebuild the index.

Our knowledge base has three layers:
- **Brand summaries** – one document per car brand with dataset statistics.
- **Buying tips** – ten general guidelines for purchasing a used car.
- **Feature explanations** – plain-English descriptions of every ML feature.

In [ ]:
# Load brand encoder so we can map Brand integer codes to brand name strings
brand_encoder = joblib.load('../models/brand_encoder.pkl')
brand_names   = list(brand_encoder.classes_)

# Load feature column order used by the price predictor
with open('../data/processed/feature_columns.json') as f:
    feature_columns = json.load(f)

print(f'Brand encoder loaded: {len(brand_names)} brands')
print('Brands:', brand_names)

In [ ]:
def most_common_fuel(sub_df):
    """Return the most common fuel type label for a brand subset."""
    diesel = sub_df['Fuel_Type_Diesel'].sum()
    petrol = sub_df['Fuel_Type_Petrol'].sum()
    other  = len(sub_df) - diesel - petrol
    if diesel >= petrol and diesel >= other:
        return 'Diesel'
    elif petrol >= other:
        return 'Petrol'
    return 'Other/CNG'


# Build one summary document per brand using dataset statistics
brand_docs = []
for brand_idx, brand_name in enumerate(brand_names):
    sub = df[df['Brand'] == brand_idx]
    if len(sub) == 0:
        continue
    avg_price = sub['Price'].mean()
    avg_seats = sub['Seats'].mean()
    fuel      = most_common_fuel(sub)
    auto_pct  = sub['Transmission_Automatic'].mean() * 100
    count     = len(sub)
    text = (
        f"The brand {brand_name} has {count} cars in our used-car dataset. "
        f"Average resale price: {avg_price:.1f} Lakh INR. "
        f"Most common fuel type: {fuel}. "
        f"Average seating capacity: {avg_seats:.1f} seats. "
        f"Automatic transmission share: {auto_pct:.0f}%."
    )
    brand_docs.append(text)

print(f'Brand documents created: {len(brand_docs)}')
print('\nSample document:')
print(brand_docs[0] if brand_docs else 'No brand documents found.')

In [ ]:
# 10 general car-buying tips
buying_tips = [
    "Tip 1: Always check the service history of a used car. A well-maintained car with regular service records is far more reliable than one with no documentation.",
    "Tip 2: Get an independent pre-purchase inspection from a trusted mechanic before buying. It can reveal hidden issues like engine wear, rust, or accident damage.",
    "Tip 3: Choose first-owner cars when possible. First-owner cars tend to be better maintained and have more predictable history than second or third-owner vehicles.",
    "Tip 4: Compare the asking price against market averages for the same brand, age, and mileage. Our ML model can give you an estimated fair price.",
    "Tip 5: Petrol cars are generally cheaper to buy but cost more to fuel for high-mileage drivers. Diesel cars are more efficient on long highway trips but have higher maintenance costs.",
    "Tip 6: Prefer lower kilometre-driven vehicles. An average Indian car is driven around 12,000 km per year. More than 15,000 km per year indicates heavy use.",
    "Tip 7: Check for any outstanding loans or insurance claims on the car using the vehicle registration number before purchase.",
    "Tip 8: Automatic transmission cars command a 10 to 20% premium over manual but are preferred in city traffic with heavy stop-and-go driving.",
    "Tip 9: Avoid buying a car more than 10 years old for daily use, as spare parts become scarce and resale value drops sharply after this point.",
    "Tip 10: Negotiate confidently. The listed resale price is usually 5 to 15% higher than the seller's true lowest price, especially for cars older than 5 years."
]

print(f'Buying tips created: {len(buying_tips)}')

In [ ]:
# Feature explanations: plain-English descriptions of every ML feature
feature_explanations = [
    "Kilometers Driven: The total distance a car has been driven since manufacture, measured in kilometres. Lower is generally better. More km means more wear on engine, transmission, and suspension. An average Indian car covers 10,000 to 15,000 km per year.",
    "Mileage (kmpl or km per kg): Fuel efficiency of the car, measuring how many kilometres it travels per litre of fuel. Higher mileage means lower running costs. Diesel cars typically achieve 18 to 25 kmpl; petrol cars 12 to 18 kmpl.",
    "Engine CC (cubic centimetres): The total volume of all engine cylinders. Larger CC engines produce more power but consume more fuel. Sub-1000cc engines are city cars; 1500 to 2000cc is the sweet spot for Indian conditions; above 2500cc is performance or SUV territory.",
    "Power BHP (brake horsepower): Maximum power output of the engine. More BHP means faster acceleration. City commuters need 60 to 100 BHP; highway cruisers benefit from 100 to 150 BHP; sports cars have 200 or more BHP.",
    "Seats: The seating capacity of the car. Standard sedans and hatchbacks seat 5; compact SUVs can seat 5 to 7; MPVs and large SUVs seat 7 to 8. More seats generally means higher price.",
    "Car Age (years): The number of years since the car was manufactured, calculated as current year minus manufacturing year. Newer cars are more expensive but more reliable. Most car insurance and loans require the car to be under 10 years old.",
    "Fuel Type: The type of fuel the car uses, such as Petrol, Diesel, CNG, LPG, or Electric. Petrol is most common in India for small cars; diesel for large cars and SUVs; CNG for fleet or taxi use. Electric vehicles are growing in cities.",
    "Transmission Type: Manual transmission requires the driver to change gears; Automatic does it automatically. Automatic cars are easier in city traffic but are heavier and more expensive. Modern automatics like CVT and DCT have largely closed the fuel efficiency gap with manual.",
    "Owner Type: Whether the current seller is the first, second, third, or fourth or higher owner. First-owner cars are preferred because they have a single trackable history. Each subsequent owner typically adds 5 to 15% depreciation.",
    "Price in Lakh INR: The resale price of the car in Indian Rupees in units of Lakh, where 1 Lakh equals 100,000 rupees. Budget used cars are under 5 Lakh; mid-range are 5 to 15 Lakh; premium are 15 to 30 Lakh; luxury are above 30 Lakh.",
    "Brand: The manufacturer of the car. In India, Maruti Suzuki holds the largest market share, followed by Hyundai, Tata, Honda, and Toyota. Premium brands include BMW, Mercedes-Benz, and Audi. Brand affects both initial price and resale value.",
    "ResNet18: The convolutional neural network architecture used in Block 1 of this project to classify car brand from a photograph. It was pre-trained on ImageNet and fine-tuned on our car brand dataset using transfer learning.",
    "Random Forest: An ensemble of decision trees used in Block 2 to predict used car prices. Each tree votes on a price and the average is taken. Random Forests handle non-linear relationships between features well and are robust to outliers.",
    "RAG (Retrieval-Augmented Generation): A technique that combines a retrieval system (FAISS vector search) with a generative language model (GPT-3.5-turbo). The retrieval step finds relevant facts; the generation step uses those facts to produce an accurate natural language answer.",
    "Transfer Learning: A machine learning technique where a model trained on one large task is adapted for a smaller related task. In Block 1, ResNet18 trained on ImageNet is fine-tuned to classify car brands, inheriting powerful visual features without training from scratch.",
]

print(f'Feature explanation documents: {len(feature_explanations)}')

In [ ]:
# Combine all documents into one knowledge base
documents = brand_docs + buying_tips + feature_explanations

print(f'Total documents in knowledge base: {len(documents)}')
print(f'  Brand summaries:       {len(brand_docs)}')
print(f'  General buying tips:   {len(buying_tips)}')
print(f'  Feature explanations:  {len(feature_explanations)}')
print()
print('Sample buying tip:')
print(buying_tips[4])

---
## Section 3 – Create Embeddings & Vector Store

### What are embeddings?

An **embedding** is a dense numerical vector (e.g. 384 floating-point numbers) that captures the *semantic meaning* of a piece of text. Two sentences that mean the same thing — even if worded differently — will have embeddings that point in similar directions in vector space.

We use `all-MiniLM-L6-v2` from the `sentence-transformers` library. It is a small, fast transformer model (22M parameters) that produces high-quality 384-dimensional embeddings, trained to make semantically similar sentences close together in embedding space.

### What is FAISS?

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search over large collections of vectors. Given a query vector, FAISS finds the *k* most similar vectors in milliseconds — even across millions of documents.

### Why cosine similarity?

**Cosine similarity** measures the angle between two vectors, not their magnitude. This is ideal for text embeddings because we care about *direction* (meaning) not *length* (word count). We achieve cosine similarity with FAISS by first L2-normalising all vectors, then using an inner-product index (`IndexFlatIP`): the inner product of two unit vectors equals cosine similarity.

In [ ]:
# Load embedding model (downloads on first run, cached locally afterwards)
print('Loading sentence-transformer model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
EMBED_DIM = 384  # output dimension of all-MiniLM-L6-v2
print(f'Embedding model loaded. Output dimension: {EMBED_DIM}')

In [ ]:
# Embed all documents using batch encoding
print(f'Embedding {len(documents)} documents...')
doc_embeddings = embedder.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

# L2-normalise so that inner product == cosine similarity
faiss.normalize_L2(doc_embeddings)

print(f'Embeddings shape: {doc_embeddings.shape}')

In [ ]:
# Build FAISS index using inner product (cosine similarity on normalised vectors)
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(doc_embeddings)

print(f'FAISS index built. Total vectors indexed: {faiss_index.ntotal}')

In [ ]:
def retrieve(query: str, k: int = 3) -> list:
    """Return the top-k documents most relevant to query using cosine similarity."""
    # Embed and normalise the query
    q_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_vec)

    # Search the FAISS index for the k nearest neighbours
    scores, indices = faiss_index.search(q_vec, k)

    return [documents[i] for i in indices[0] if i < len(documents)]

In [ ]:
# Test retrieval with 3 example queries
test_queries = [
    'What is the average price of BMW cars?',
    'What should I look for when buying a used car?',
    'What does engine CC mean?'
]

for query in test_queries:
    print(f'\nQuery: "{query}"')
    print('-' * 60)
    for j, doc in enumerate(retrieve(query, k=3), 1):
        print(f'  [{j}] {doc[:120]}...')

---
## Section 4 – Prompt Engineering & LLM Integration

### The RAG Pipeline

Our pipeline has three stages:

```
User question
     |
     v
1. RETRIEVE  -- embed query -> FAISS search -> top-3 context documents
     |
     v
2. AUGMENT   -- build structured prompt with XML tags:
               <context>...retrieved docs...</context>
               <question>...user question...</question>
     |
     v
3. GENERATE  -- GPT-3.5-turbo reads the prompt -> produces grounded answer
```

### Why XML tags in prompts?

Structured tags (`<context>`, `<question>`) act as clear separators that help the LLM distinguish between:
- **Retrieved facts** it should treat as ground truth.
- **The user's actual question** it should answer.

This reduces hallucination: the model is anchored to the provided context rather than improvising from its weights.

In [ ]:
def ask_car_advisor(question: str, context_docs: list) -> str:
    """
    Call GPT-3.5-turbo with a structured prompt using XML tags.

    Parameters
    ----------
    question     : the user's question
    context_docs : list of retrieved knowledge-base documents

    Returns
    -------
    str  the model's answer
    """
    context_text = '\n\n'.join(context_docs)

    system_msg = (
        'You are an expert car advisor assistant specialising in the Indian used-car market. '
        'Provide concise, accurate, and helpful answers based on the provided context.'
    )

    # XML tags clearly separate retrieved facts from the user question
    user_msg = (
        f'<context>\n{context_text}\n</context>\n\n'
        f'<question>\n{question}\n</question>\n\n'
        'Answer based on the context above. Be concise and helpful. '
        'If the context does not contain enough information, say so briefly and share general knowledge.'
    )

    try:
        response = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[
                {'role': 'system', 'content': system_msg},
                {'role': 'user',   'content': user_msg}
            ],
            max_tokens=300,
            temperature=0.3  # lower temperature = more factual, less creative
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'[API Error] {e}'

In [ ]:
def rag_answer(question: str, k: int = 3) -> str:
    """Full RAG pipeline: retrieve -> augment prompt -> generate answer."""
    context_docs = retrieve(question, k=k)
    return ask_car_advisor(question, context_docs)

In [ ]:
# Test the full RAG pipeline with 5 different questions
rag_questions = [
    'Which used car brand offers the best value for money in India?',
    'Is it worth buying a diesel car in 2024?',
    'What is the difference between manual and automatic transmission?',
    'How many kilometres is too many for a used car?',
    'Why do first-owner cars cost more than second-owner cars?'
]

for q in rag_questions:
    print(f'\nQ: {q}')
    print(f'A: {rag_answer(q)}')
    print('=' * 70)

---
## Section 5 – Prompt Comparison (Course Requirement)

### Why compare prompt strategies?

The quality of an LLM response depends heavily on *how* you frame the prompt. This section compares three strategies on the same question to demonstrate what RAG and structured prompting add over a plain API call.

| Strategy | System role | RAG context | XML tags |
|----------|------------|-------------|----------|
| **A – Bare question** | No | No | No |
| **B – RAG only** | No | Yes | No |
| **C – Full RAG** | Yes | Yes | Yes |

In [ ]:
COMPARISON_QUESTION = 'Should I buy a petrol or diesel car?'

# Retrieve context once, reuse for strategies B and C
context_docs = retrieve(COMPARISON_QUESTION, k=3)
context_text = '\n\n'.join(context_docs)

print(f'Question: "{COMPARISON_QUESTION}"')
print(f'Retrieved {len(context_docs)} context documents.')

In [ ]:
# Strategy A: Bare question -- no system role, no RAG context
try:
    resp_a = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': COMPARISON_QUESTION}],
        max_tokens=200,
        temperature=0.5
    )
    answer_a = resp_a.choices[0].message.content.strip()
except Exception as e:
    answer_a = f'[Error] {e}'

print('=== Strategy A: Bare question (no role, no context) ===')
print(answer_a)

In [ ]:
# Strategy B: RAG context only -- no system role, no XML tags
user_msg_b = (
    f'Here is some background information:\n{context_text}\n\n'
    f'Question: {COMPARISON_QUESTION}'
)

try:
    resp_b = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': user_msg_b}],
        max_tokens=200,
        temperature=0.5
    )
    answer_b = resp_b.choices[0].message.content.strip()
except Exception as e:
    answer_b = f'[Error] {e}'

print('=== Strategy B: RAG context, no system role, no XML tags ===')
print(answer_b)

In [ ]:
# Strategy C: Full RAG -- system role + XML tags (our final approach)
try:
    answer_c = ask_car_advisor(COMPARISON_QUESTION, context_docs)
except Exception as e:
    answer_c = f'[Error] {e}'

print('=== Strategy C: Full RAG with system role + XML tags ===')
print(answer_c)

### Analysis: Which prompt strategy is best and why?

**Strategy A (bare question)** gives a generic, text-book answer about petrol vs diesel. The model draws entirely from its training data. There is no grounding in our specific dataset — it cannot mention Indian market specifics or dataset-specific price ranges.

**Strategy B (RAG context, no structure)** injects our retrieved knowledge but without a system role or structured tags. The model may mix the injected context with its own opinions, or fail to clearly prioritise the retrieved facts over its prior knowledge. The boundary between background reading and the actual question is unclear.

**Strategy C (full RAG with system role + XML tags)** is clearly the best because:
1. **System role** sets the persona and tone — the model knows it is an expert car advisor for the Indian market, not a general chatbot.
2. **XML tags** (`<context>`, `<question>`) create clear semantic boundaries: the model distinguishes between ground truth and what it must answer.
3. **Retrieved context** anchors the answer to our dataset — making it specific, verifiable, and free of hallucinated statistics.

**Conclusion:** RAG alone improves factual grounding; prompt structure (role + XML tags) improves focus and reduces hallucination. Together they produce the highest quality answer.

---
## Section 6 – Full Pipeline Integration

### How the three blocks work together

This section ties all three notebooks into one end-to-end function:

```
INPUT:  car_image.jpg  +  user-provided specs (year, km, fuel, ...)
         |
         |-- Block 1 (CV / ResNet18)       --> predicted brand name
         |
         |-- Block 2 (ML / Random Forest)  --> predicted price (Lakh INR)
         |
         |-- Block 3 (NLP / RAG)           --> natural language explanation
                                                    |
OUTPUT: (brand, price, explanation)
```

`full_pipeline()` is designed to be directly importable by the Streamlit app — it returns a clean tuple of results without any side effects.

In [ ]:
# Load CV model (ResNet18) -- same architecture as 02_cv_model.ipynb
with open('../models/class_names.json') as f:
    cv_class_names = json.load(f)   # list[str]: index -> US brand name
NUM_CV_CLASSES = len(cv_class_names)

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

cv_device = get_device()

# Reconstruct the identical architecture used when the weights were saved
try:
    cv_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except AttributeError:
    cv_model = models.resnet18(pretrained=False)

cv_model.fc = nn.Linear(cv_model.fc.in_features, NUM_CV_CLASSES)
cv_model.load_state_dict(torch.load('../models/car_brand_classifier.pth', map_location=cv_device))
cv_model = cv_model.to(cv_device)
cv_model.eval()

print(f'CV model loaded: ResNet18 ({NUM_CV_CLASSES} classes, device={cv_device})')

In [ ]:
# Load ML models: price predictor + scaler (brand_encoder already loaded in Section 2)
price_predictor = joblib.load('../models/price_predictor.pkl')
scaler          = joblib.load('../models/scaler.pkl')

print(f'Price predictor: {type(price_predictor).__name__}')
print(f'Scaler:          {type(scaler).__name__}')
print(f'Brand encoder:   {len(brand_encoder.classes_)} brands')

In [ ]:
# Image preprocessing (identical to val_transform in 02_cv_model.ipynb)
_cv_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def predict_brand_from_image(image_path: str):
    """Run CV model on an image. Returns (brand_name, confidence_pct)."""
    img    = Image.open(image_path).convert('RGB')
    tensor = _cv_transform(img).unsqueeze(0).to(cv_device)
    with torch.no_grad():
        logits = cv_model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
    idx        = probs.argmax().item()
    confidence = probs[idx].item() * 100
    return cv_class_names[idx], round(confidence, 1)


def predict_price_from_features(
        brand: str, year: int, km_driven: float,
        fuel_type: str, transmission: str,
        mileage: float, engine: float, power: float,
        seats: int, owner_type: str = 'First') -> float:
    """
    Run ML price predictor. Returns predicted price in Lakh INR.
    Mirrors the predict_price() function from 03_ml_numeric.ipynb.
    """
    car_age = 2024 - year

    # Scale numeric features (order must match scaler fit order)
    raw_num = np.array([[km_driven, engine, power, mileage, car_age]], dtype=float)
    scaled  = scaler.transform(raw_num)[0]
    km_s, eng_s, pwr_s, mil_s, age_s = scaled

    # Encode brand -- CV predicts US brands; ML encoder has Indian brands.
    # Use exact match if possible; fall back to median brand index.
    try:
        brand_enc = float(brand_encoder.transform([brand])[0])
    except ValueError:
        brand_enc = float(len(brand_encoder.classes_) // 2)

    row = pd.DataFrame([{
        'Kilometers_Driven':         km_s,
        'Mileage':                   mil_s,
        'Engine':                    eng_s,
        'Power':                     pwr_s,
        'Seats':                     float(seats),
        'Brand':                     brand_enc,
        'Car_Age':                   age_s,
        'Fuel_Type_Diesel':          float(fuel_type == 'Diesel'),
        'Fuel_Type_Petrol':          float(fuel_type == 'Petrol'),
        'Transmission_Automatic':    float(transmission == 'Automatic'),
        'Transmission_Manual':       float(transmission == 'Manual'),
        'Owner_Type_First':          float(owner_type == 'First'),
        'Owner_Type_Fourth & Above': float(owner_type == 'Fourth & Above'),
        'Owner_Type_Second':         float(owner_type == 'Second'),
        'Owner_Type_Third':          float(owner_type == 'Third'),
    }])[feature_columns]  # enforce column order from feature_columns.json

    return round(float(price_predictor.predict(row)[0]), 2)


print('Helper functions defined.')

In [ ]:
def full_pipeline(
        image_path: str,
        year: int,
        km_driven: float,
        fuel_type: str,
        transmission: str,
        mileage: float,
        engine: float,
        power: float,
        seats: int,
        owner_type: str = 'First'
):
    """
    End-to-end pipeline: image + specs -> brand + price + explanation.

    Parameters
    ----------
    image_path   : path to car image file
    year         : manufacturing year (e.g. 2018)
    km_driven    : total km on the odometer (e.g. 45000)
    fuel_type    : 'Petrol', 'Diesel', 'CNG', 'LPG', or 'Electric'
    transmission : 'Manual' or 'Automatic'
    mileage      : fuel efficiency in kmpl (e.g. 18.5)
    engine       : engine displacement in CC (e.g. 1497)
    power        : max power in BHP (e.g. 108.5)
    seats        : seating capacity (e.g. 5)
    owner_type   : 'First', 'Second', 'Third', 'Fourth & Above'

    Returns
    -------
    tuple: (predicted_brand: str, predicted_price: float, explanation: str)
    """
    # Step 1: CV model predicts brand from the car image
    predicted_brand, confidence = predict_brand_from_image(image_path)

    # Step 2: ML model predicts resale price from tabular features
    predicted_price = predict_price_from_features(
        predicted_brand, year, km_driven, fuel_type, transmission,
        mileage, engine, power, seats, owner_type
    )

    # Step 3: RAG generates a natural language explanation and recommendation
    rag_query = (
        f'Tell me about a {year} {predicted_brand} with {km_driven:.0f} km driven, '
        f'{fuel_type} fuel, {transmission} transmission, and an estimated price of '
        f'{predicted_price} Lakh INR. Should I buy it?'
    )
    retrieved = retrieve(rag_query, k=3)

    explanation_prompt = (
        f'Our AI pipeline analysed a car image and predicted the following results:\n'
        f'- Brand identified by CV model: {predicted_brand} (confidence: {confidence:.1f}%)\n'
        f'- Estimated resale price by ML model: {predicted_price} Lakh INR\n'
        f'- Year: {year} | Km driven: {km_driven:.0f} | Fuel: {fuel_type} | '
        f'Transmission: {transmission} | Engine: {engine} CC | Power: {power} BHP\n\n'
        f'Based on this information and the context provided, give a concise buying recommendation.'
    )
    explanation = ask_car_advisor(explanation_prompt, retrieved)

    return predicted_brand, predicted_price, explanation


print('full_pipeline() function defined and ready.')

In [ ]:
# Test full_pipeline with one example image from the dataset
img_dir    = '../data/raw/Auto_Bilder'
all_images = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
test_image = all_images[0] if all_images else None

if test_image is None:
    print('No images found in Auto_Bilder directory. Skipping pipeline test.')
else:
    print(f'Test image: {os.path.basename(test_image)}')
    print('Running full pipeline (CV -> ML -> RAG)...\n')

    predicted_brand, predicted_price, explanation = full_pipeline(
        image_path   = test_image,
        year         = 2018,
        km_driven    = 42000,
        fuel_type    = 'Petrol',
        transmission = 'Manual',
        mileage      = 18.5,
        engine       = 1497.0,
        power        = 108.5,
        seats        = 5
    )

    print(f'Predicted Brand : {predicted_brand}')
    print(f'Predicted Price : {predicted_price} Lakh INR')
    print(f'\nExplanation:\n{explanation}')

---
## Section 7 – Save RAG Components

We persist the three RAG artefacts so the Streamlit app can load them without re-running this notebook. Embedding 60+ documents takes around 10 seconds on CPU — saving avoids this cost at inference time.

Saved artefacts:
1. **`faiss_index.bin`** — the FAISS index with all document embeddings.
2. **`rag_documents.json`** — the list of raw document texts (needed to return the actual text after FAISS returns integer indices).
3. **`rag_config.json`** — metadata: embedding model name, number of documents, embedding dimension, default retrieval k.

In [ ]:
os.makedirs('../models', exist_ok=True)

# 1. FAISS index
faiss_path = '../models/faiss_index.bin'
faiss.write_index(faiss_index, faiss_path)
print(f'FAISS index saved       ->  {faiss_path}')

# 2. Documents list
docs_path = '../models/rag_documents.json'
with open(docs_path, 'w', encoding='utf-8') as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)
print(f'RAG documents saved     ->  {docs_path}')

# 3. Config metadata
rag_config = {
    'embedding_model': 'all-MiniLM-L6-v2',
    'embedding_dim':   EMBED_DIM,
    'k_default':       3,
    'num_documents':   len(documents),
    'index_type':      'IndexFlatIP',
    'openai_model':    'gpt-3.5-turbo',
    'temperature':     0.3
}
config_path = '../models/rag_config.json'
with open(config_path, 'w') as f:
    json.dump(rag_config, f, indent=2)
print(f'RAG config saved        ->  {config_path}')

print()
print('All artefacts saved. Load in app.py with:')
print('  import faiss, json')
print('  faiss_index = faiss.read_index("models/faiss_index.bin")')
print('  documents   = json.load(open("models/rag_documents.json"))')
print('  rag_config  = json.load(open("models/rag_config.json"))')

In [ ]:
# Verify saved files
for path in [faiss_path, docs_path, config_path]:
    size_kb = os.path.getsize(path) / 1024
    print(f'{path}  ({size_kb:.1f} KB)')

print('\nNotebook 04 complete. RAG pipeline is ready for Streamlit.')